# Lab 7 — Judges, Agreement, and Rankings

**MA553 · Session 8**

Name:

---

In this lab you work with the MT-Bench votes from the session. Each vote compares the
answers of two models to the same question and turn. The judge chooses A, B, or a tie.

**Part A** measures how often judges agree: two people, and GPT-4 against a person.

**Part B** turns the votes of people into a ranking of the models, and checks which
differences the data support.

Work through the cells in order. Six cells are marked **Write this yourself**: A2, A3,
A4, B1, B2 and B3. Each says which variables to create. Later cells use those names.
The **Answer here** cells are short written answers.

No language model is needed. Every cell runs in a few seconds, except B3, which takes
about one minute.

# Part A — Agreement between judges

## Setup

The two vote files are in the folder `data/`. Run this.

In [ ]:
import itertools
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score

people = pd.read_parquet("data/human_votes.parquet")
gpt4 = pd.read_parquet("data/gpt4_votes.parquet")
print("Votes from people:", len(people))
print("Votes from GPT-4: ", len(gpt4))

## A1 — The data

Each row is one vote. The column `winner` holds the vote: `model_a`, `model_b`, or `tie`.
The columns `conversation_a` and `conversation_b` hold the text of the two answers.

Run this. It shows the votes on one pair: question 107, first turn, claude-v1 against
gpt-3.5-turbo (the pair of slide 5).

In [ ]:
pair = ["claude-v1", "gpt-3.5-turbo"]
cols = ["question_id", "turn", "model_a", "model_b", "judge", "winner"]
same_pair = lambda d: (d.question_id == 107) & (d.turn == 1) & d.model_a.isin(pair) & d.model_b.isin(pair)
print(people[same_pair(people)][cols].to_string(index=False))
print()
print(gpt4[same_pair(gpt4)][cols].to_string(index=False))

**Answer here.** In the first row, which model was answer A? Which model did that person
choose?

*Answer here.*

## Comparisons between two people

Many pairs were judged by more than one person. This cell takes every two votes on the
same pair (same question, models, and turn). For each such comparison, it stores the vote
of the first person in `first` and the vote of the second person in `second`.

Run this.

In [ ]:
first, second = [], []
for _, group in people.groupby(["question_id", "model_a", "model_b", "turn"]):
    for v1, v2 in itertools.combinations(group.winner, 2):
        first.append(v1)
        second.append(v2)
first, second = np.array(first), np.array(second)
print("Comparisons between two people:", len(first))

## A2 — Kappa for two people ✏️ Write this yourself

**Goal.** Compute Cohen's kappa by hand, as on the slides.

1. `P_o`: the share of comparisons where `first` equals `second`.
2. For each vote `c` in `["model_a", "model_b", "tie"]`, the share of `first` equal to
   `c` and the share of `second` equal to `c`. Print them.
3. `P_e`: the sum over the three votes of (share for `first`) × (share for `second`).
4. `kappa = (P_o - P_e) / (1 - P_e)`. Check it with `cohen_kappa_score(first, second)`.

**Expected.** `P_o` = 0.646, `P_e` = 0.348, kappa = 0.46.

In [ ]:
# Your code here.

## A3 — GPT-4 against a person ✏️ Write this yourself

This cell matches each vote of a person with the GPT-4 vote on the same pair. GPT-4's
`tie (inconsistent)` counts as a tie. Run it first.

In [ ]:
gpt4["vote"] = gpt4.winner.replace({"tie (inconsistent)": "tie"})
matched = people.merge(gpt4[["question_id", "model_a", "model_b", "turn", "vote"]],
                       on=["question_id", "model_a", "model_b", "turn"])
person, judge = matched.winner.values, matched.vote.values
print("Comparisons between GPT-4 and a person:", len(matched))

**Goal.** Repeat A2 with `person` and `judge` in place of `first` and `second`. Store the
results in `P_o_g`, `P_e_g` and `kappa_g`, and print them.

**Expected.** `P_o_g` = 0.673, `P_e_g` = 0.464, kappa = 0.39.

In [ ]:
# Your code here.

**Answer here.** GPT-4 and a person give the same vote more often than two people do, but
their kappa is lower. Use your numbers for `P_e` to explain why.

*Answer here.*

## A4 — Order sensitivity ✏️ Write this yourself

GPT-4 judged each pair in both orders. When the two orders gave different winners, the
vote was recorded as `tie (inconsistent)`.

**Goal.** Count the GPT-4 votes in each value of `winner` (use `value_counts()`), and print
the share of `tie (inconsistent)`.

**Expected.** 380 of 2,400 votes (15.8%) are `tie (inconsistent)`.

In [ ]:
# Your code here.

# Part B — From votes to a ranking

From here on, we use the votes of people only.

## B1 — Win rates ✏️ Write this yourself

**Goal.** For each model, compute its win rate: the share of its comparisons that it won,
with a tie counting as half a win. Store the result in a dictionary `win_rate`, and print
the models from highest to lowest win rate.

Hint: loop over the rows of `people`. Each row gives one comparison to `model_a` and one
to `model_b`.

**Expected.** gpt-4 73.3%, gpt-3.5-turbo 65.2%, claude-v1 65.0%, vicuna-13b-v1.2 48.0%,
alpaca-13b 28.7%, llama-13b 15.4%.

In [ ]:
# Your code here.

## B2 — Bradley–Terry ratings ✏️ Write this yourself

**Goal.** Fit the Bradley–Terry model as a logistic regression, as on the slides.

1. Build the data: one row per vote, one column per model, with +1 for `model_a`, −1 for
   `model_b`, and 0 for the others. The outcome is 1 when `model_a` won and 0 when
   `model_b` won. A tie becomes two rows with the same columns: one with outcome 1, one
   with outcome 0.
2. Fit `LogisticRegression(fit_intercept=False, C=1e6)`. The coefficients
   (`.coef_[0]`) are the strengths.
3. Subtract the strength of llama-13b from every strength, so that llama-13b is at 0.
4. Convert to ratings: `1000 + 400 / np.log(10) * strength` (about 174 times the strength). Store them in a dictionary `rating` and
   print them from highest to lowest.

Write it as a function `fit_ratings(votes)` that takes a table of votes and returns
`rating`. B3 uses this function.

**Expected.** gpt-4 1342, claude-v1 1290, gpt-3.5-turbo 1283, vicuna-13b-v1.2 1193,
alpaca-13b 1095, llama-13b 1000.

In [ ]:
# Your code here.

**Answer here.** In B1, gpt-3.5-turbo has a slightly higher win rate than claude-v1. In
B2, claude-v1 has the higher rating. How can the two orders differ?

*Answer here.*

## B3 — Can we order claude-v1 and gpt-3.5-turbo? ✏️ Write this yourself

**Goal.** Compute a 95% bootstrap interval for the difference in rating between claude-v1
and gpt-3.5-turbo.

1. Resample the questions: draw 80 question ids with replacement from the unique values
   of `people.question_id`. Keep all the votes on each drawn question, so both turns stay
   together. Use `rng = np.random.default_rng(7)`.
2. Fit the ratings on the resampled votes with `fit_ratings`, and store
   rating of claude-v1 − rating of gpt-3.5-turbo.
3. Repeat 200 times. Print the 2.5th and 97.5th percentiles of the 200 differences.

**Expected.** An interval of about −30 to +40 points. Your values can differ by a few
points, because the resamples are random.

In [ ]:
# Your code here.

**Answer here.** (1) Does the interval support an order between the two models? (2) Why do
we resample questions and not single votes?

*Answer here.*

## B4 — How many questions?

The number of independent comparisons needed for 80% power, with a two-sided test at the
5% level, is given by the formula of the slides. Run this.

In [ ]:
from scipy.stats import norm

def comparisons_needed(p, power=0.8):
    za, zb = norm.ppf(0.975), norm.ppf(power)
    return int(np.ceil(((za * 0.5 + zb * np.sqrt(p * (1 - p))) / (p - 0.5)) ** 2))

for p in (0.55, 0.60, 0.65, 0.70):
    print(f"true win rate {p:.0%}: {comparisons_needed(p)} comparisons")

**Answer here.** A company wants to show that its new model beats the old one, with a true
win rate of about 55%. It plans a benchmark of 100 questions with one turn each. What would
you tell them?

*Answer here.*

---
**Before you submit.** Restart the kernel and run every cell from top to bottom.